# Olympus OM-D E-M1x GPS log evaluation

In [5]:
# imports
from os import getenv
from pathlib import Path
from dotenv import load_dotenv
import ipyleaflet as ipylf
import pynmea2 as nmea
import pandas as pd
from datetime import datetime as dt
from geodata import GeoDataPoint as gdp

In [6]:
load_dotenv()

GpsLogPath = getenv("GPS_LOG_PATH")
GpsLogPath = Path(GpsLogPath).resolve()

SensorLogPath = getenv("SENSOR_LOG_PATH")
SensorLogPath = Path(SensorLogPath).resolve()

print(GpsLogPath)
print(SensorLogPath)

/home/davo/git-clones/Maps/E-M1xGPS/GPSLOG/*
/home/davo/git-clones/Maps/E-M1xGPS/SNSLOG/*


In [ ]:
# basic map initialization

center = (52.283935, 8.022916)
map = ipylf.Map(center=center, zoom=13)

marker = ipylf.Marker(location=center)
map.add_control(marker)

display(map)

In [ ]:
# parse Olympus GPS Log data

with open(GpsLogPath) as GpsLogFile:
    lines = GpsLogFile.readlines()

data = []

for line in lines:
    try:
        message = nmea.parse(line)

        if message.sentence_type == "RMC":
            dateTime = dt.combine(message.datestamp, message.timestamp)
            
            geodatapoint = gdp(dateTime, message.latitude, message.longitude)
            
            data.append(geodatapoint)

    except nmea.ParseError:
        continue

In [ ]:
# add GPS data to map
for datapoint in data:
	marker = ipylf.Marker(location=datapoint.coordinate, draggable=False, title=f"Lat: {datapoint.latitude}, Lon: {datapoint.longitude}")
	map.add_control(marker)

display(map)

In [ ]:
# parse Olympus sensor data

for file in SensorLogPath:

    rows = []
    with open(file) as sensorLogFile:
        current_time = None
        for line in sensorLogFile:
            line = line.strip()
            # Time
            if line.startswith("$OLTIM"):
                _, date, time = line.split(",")
                current_time = dateTime.strptime(date + time, "%Y%m%d%H%M%S")
            # Compass
            elif line.startswith("$OLCMP"):
                rows[0]["compass_deg"] = float(line.split(",")[1])
            # Pressure
            elif line.startswith("$OLPRE"):
                p = line.split(",")
                rows[0]["pressure_hpa"] = float(p[1])
            # Temperature
            elif line.startswith("$OLTMP"):
                t = line.split(",")
                rows[0]["temp_c"] = float(t[1])
            # Accelerometer
            elif line.startswith("$OLACC"):
                a = line.split(",")
                rows[0]["acc_x"] = float(a[1])
                rows[0]["acc_y"] = float(a[2])
                rows[0]["acc_z"] = float(a[3])
            # Start a new record
            elif line.startswith("$OLTIM"):
                rows.append({"datetime": current_time})

dataframe = pd.DataFrame(rows)
dataframe.head()